## Find neighbours of atoms in PDB structures using biopython

The PDB structure contains the (x, y, z) coordinates of every atom in the structure. biopython has an implementation of a fast algorithm, `NeighborSearch`, that can find neighbouring atoms that are within a certain distance of a given coordinate.

In [4]:
import os.path
from Bio.PDB import PDBParser, NeighborSearch

In [5]:
# loa versucht 
import os
from Bio.PDB import PDBParser, NeighborSearch

path = os.path.join("data", "output.txt")
if os.path.exists(path):
    print("File available:", path)

Define the directory where the Chothia renumbered PDB file are stored.

In [6]:
PDB_DIR = "../data cleanup/pdb_cf"




Given the `pdb_id`, find the path to the corresponding PDB file

In [7]:
pdb_id = "6xc3"
filename = os.path.join(PDB_DIR, f"{pdb_id}.pdb")

First, we need to parse the PDB file

In [8]:
from Bio.PDB import PDBParser

parser = PDBParser(PERMISSIVE=1)
structure = parser.get_structure(pdb_id, filename)

Then we need to initialise the data structure that is used to perform the search of neighbours. Here we use all atoms of the structure for demonstration. This can be wasteful. Later, it can be advantageous to only use the atoms of the antigen, for example.

In [9]:
atoms = list(structure.get_atoms())
ns = NeighborSearch(atoms)

Now we can use the `ns.search(coords, distance)` method to find all atoms that are within `distance` angstroms from the position specified by the `[x, y, z]` array `coords`.

We can retrieve the position of an atom using its `coord` property:

In [10]:
atom = structure[0]['H'][52]['N']
atom.coord

array([-60.134,  62.357,  23.903], dtype=float32)

To find all atoms in vicinity of an atom we simply pass the coordinates of the atom to the `ns.search` method.

In [11]:
close_atoms = ns.search(atom.coord, 4.0)

`close_atoms` now contains the atom entities of all atoms that are within 4.0 angstrom of the supplied coordinates. To find the residues and chains these atoms belong to, we can either use repeated application of the `get_parent()` method, or we can simply use the `get_full_id()` method.

As we want the output as a DataFrame, we first create a vector of dicts, and then create the DataFrame. 

In [12]:
import pandas as pd

data = []

for atom in close_atoms:
    s, m, c, r, a = atom.full_id
    resnum = str(r[1]) + r[2]

    # to get the resname we need to access the residue the atom belongs to
    residue = atom.get_parent()
    resname = residue.get_resname()
    data.append(dict(chain = c, resnum = resnum, resname = resname, atom = a[0]))


df = pd.DataFrame(data)

df

,chain,resnum,resname,atom
0,H,51,ILE,N
1,H,51,ILE,O
2,H,52A,PRO,CD
3,H,33,TRP,CE3
4,H,52,TYR,CG
5,H,52,TYR,CB
6,H,52,TYR,CA
7,H,51,ILE,C
8,H,51,ILE,CG2
9,H,52,TYR,N


Now we have learned how to find all atoms within the vicinity of a given atom. 

We can use this to find the atomic contact points between residues of an antibody and residues of its bound antigen.
We say that the residue of an antibody is in contact with the residue of an antigen if the distance 
of the corresponding atoms is less that a threshold distance. Often a threshold of 4 angstroms is used.

Write a function `atomic_contact_points(ab_chain, ag_chain, distance)` that loops over the atoms in ab_chain to get all atoms of ag_chain that are within distance, and reports a DataFrame with columns
- ab_resnum
- ab_icode
- ab_resname
- ab_atom
- ag_resnum
- ag_icode
- ag_resname
- ag_atom

But only report for residues that are amino acids, i.e. het_flag == ' '.

In [13]:
def atomic_contact_points(ab_chain, ag_chain, distance):
    res = []
    ag_atoms = list(ag_chain.get_atoms())
    ns = NeighborSearch(ag_atoms)
    for ab_atom in ab_chain.get_atoms():
        ab_res = ab_atom.get_parent()
        close_ag_atoms = ns.search(ab_atom.coord, 4.0)
        for ag_atom in close_ag_atoms:
            ag_res = ag_atom.get_parent()
            if ag_res.id[0] == " " and ab_res.id[0] == " ": 
                res.append(dict(ab_resnum = ab_res.id[1],
                                ab_icode = ab_res.id[2],
                                ab_resname = ab_res.get_resname(),
                                ab_atom = ab_atom.id,
                                ag_resnum = ag_res.id[1],
                                ag_icode = ag_res.id[2],
                                ag_resname = ag_res.get_resname(),
                                ag_atom = ag_atom.id))
    return pd.DataFrame(res)
  

Now let's test the function

In [14]:
cp = atomic_contact_points(structure[0]['H'], structure[0]['C'], 4.0)
cp

,ab_resnum,ab_icode,ab_resname,ab_atom,ag_resnum,ag_icode,ag_resname,ag_atom
0,27,,TYR,C,370,,ASN,OD1
1,27,,TYR,CB,370,,ASN,OD1
2,28,,GLY,N,370,,ASN,OD1
3,28,,GLY,N,369,,TYR,O
4,28,,GLY,CA,369,,TYR,O
...,...,...,...,...,...,...,...,...
68,101,,ASP,CG,386,,LYS,CE
69,101,,ASP,CG,386,,LYS,NZ
70,101,,ASP,OD1,386,,LYS,CE
71,101,,ASP,OD2,386,,LYS,CE


Now we have a DataFrame with atomic contact points. Often we are interested in residue contact points. These can be obtained from the DataFrame using  (check the pandas grouping documentation).

For, example, to get the number of atom-atom and residue-residue contacts per antibody residue, we can use

In [15]:
(cp
.groupby(['ab_resnum', 'ab_resname'])
.agg(natomcontacts = ('ag_resnum', 'size'),
    nrescontacts = ('ag_resnum', 'nunique'))
.reset_index()
)

,ab_resnum,ab_resname,natomcontacts,nrescontacts
0,27,TYR,2,1
1,28,GLY,3,2
2,30,ILE,6,3
3,31,THR,5,2
4,52,TYR,14,3
5,54,ASP,4,1
6,56,GLU,3,1
7,96,SER,10,4
8,97,GLY,2,1
9,98,ILE,11,3


We are also interested in the occurrence of residue numbers in the heavy and light chains.
Write a function `residue_occurrence(chain)` that creates as output a DataFrame with columns 
- ab_resnum
- ab_icode
- ab_resname

Restrict to het_name == ' ' and resnum <= 128

In [16]:
def residue_occurrence(chain):

    results = []
    for residue in chain.get_residues():
       if  residue.id[0] == " " and residue.id[1] <= 128:
           results.append(dict(ab_resnum = residue.id[1],
                               ab_icode = residue.id[2],
                               ab_resname = residue.get_resname()))
    return pd.DataFrame(results) 
           



Let's test the function

In [17]:
ro = residue_occurrence(structure[0]['H'])
ro

,ab_resnum,ab_icode,ab_resname
0,1,,GLN
1,2,,MET
2,3,,GLN
3,4,,LEU
4,5,,VAL
...,...,...,...
129,124,,LEU
130,125,,ALA
131,126,,PRO
132,127,,SER


In the next notebook we are going to loop over all PDB files. It will, therefore, be necessary to add some more columns to the DataFrames. We can add a column at a given position using the `insert()` method. For example,

In [18]:
ro.insert(loc = 0, column = 'chain_type', value = 'heavy')
ro.insert(loc = 0, column = 'pdb_id', value = pdb_id)
ro

,pdb_id,chain_type,ab_resnum,ab_icode,ab_resname
0,6xc3,heavy,1,,GLN
1,6xc3,heavy,2,,MET
2,6xc3,heavy,3,,GLN
3,6xc3,heavy,4,,LEU
4,6xc3,heavy,5,,VAL
...,...,...,...,...,...
129,6xc3,heavy,124,,LEU
130,6xc3,heavy,125,,ALA
131,6xc3,heavy,126,,PRO
132,6xc3,heavy,127,,SER
